# ACM Research 2026 — Maia-3 on Colab

Track 2 uses pretrained Maia-3 5M for S8 baseline evaluation and S9 player fine-tuning.

Run sections 1–4 in every new session. Then run the sections you need. Select **Runtime → Change runtime type → T4 GPU** for full evaluation and fine-tuning. CPU also works for small checks.

The notebook uses the group's frozen data in Google Drive. Obtain access to the shared `acm_chess` folder first. A folder shared with you can be added as a shortcut in My Drive. Set `PROJ` below to the mounted path of that folder.

S8 and S9 must be available in the GitHub branch selected in section 3. If they are still local files, upload both scripts into `/content/repo/track2_finetunemaia3/` using Colab's Files panel after cloning the repo. Reconnects can remove files stored under `/content`.

## 1. Check the runtime

In [ ]:
import json
import shutil
import subprocess
import sys
from pathlib import Path

print(sys.version)
if shutil.which('nvidia-smi'):
    subprocess.run(['nvidia-smi'], check=True)
else:
    print('CPU runtime. Select a T4 GPU for the full experiment.')

## 2. Mount Google Drive

`PROJ/data` must contain the frozen `raw/` and `splits/` folders. Maia-2 predictions should be in `PROJ/baselines/` for the S8 comparison.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')
PROJ = Path('/content/drive/MyDrive/acm_chess')
DATA_ROOT = PROJ / 'data'
MAIA2_DIR = PROJ / 'baselines'
BASELINE_DIR = PROJ / 'baselines_maia3'
FINETUNE_DIR = PROJ / 'finetuned_maia3'
RUNS_DIR = PROJ / 'runs' / 'maia3'
for directory in (BASELINE_DIR, FINETUNE_DIR, RUNS_DIR):
    directory.mkdir(parents=True, exist_ok=True)
print('Shared data:', DATA_ROOT)
print('Maia-3 results:', BASELINE_DIR)

## 3. Get the project and install Maia-3

Set `REPO_REF` to the branch containing your S8 and S9 scripts. The project is cloned into `/content/repo`. The Maia-3 package is installed from its official repository along with the libraries used by S8, S9, and P8.

In [ ]:
REPO_URL = 'https://github.com/SchwiLynn/ACM_RESEARCH_2026-Training-a-chess-model.git'
REPO_REF = 'Tung'  # Branch containing the S8 and S9 scripts.
REPO = Path('/content/repo')
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', '--branch', REPO_REF, REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin'], cwd=REPO, check=True)
    subprocess.run(['git', 'checkout', REPO_REF], cwd=REPO, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', REPO_REF], cwd=REPO, check=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install',
    'git+https://github.com/CSSLab/maia3.git', 'pandas', 'numpy', 'matplotlib',
], check=True)

In [ ]:
import importlib.metadata
import torch

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
PLAYER = 'thakt'
TRAIN_FILE = 'train_20_seed0.csv'  # Use train_all.csv for the full training split.
BATCH_SIZE = 64
EPOCHS = 20
LEARNING_RATE = 1e-4
print('Device:', DEVICE)
print('Maia-3:', importlib.metadata.version('maia3'))

S8 = REPO / 'track2_finetunemaia3/s8_maia3_baseline.py'
S9 = REPO / 'track2_finetunemaia3/s9_finetune.py'
for script in (S8, S9):
    if not script.is_file():
        raise FileNotFoundError(
            f'{script} is missing. Upload S8 and S9 to its parent folder, '
            'or select their GitHub branch in REPO_REF and rerun section 3.'
        )

def run_script(script, *arguments):
    subprocess.run([sys.executable, '-u', str(script), *map(str, arguments)],
                   cwd=REPO, check=True)

## 4. Inspect the frozen data

This section reads the existing splits. Keep the same test sets used for Maia-2. Data preparation is managed by the shared pipeline P5–P7.

For each player, S8 with history needs `data/splits/<player>/test.csv` and `data/raw/<player>_rapid.pgn`. S9 also needs the selected training CSV and `val.csv`.

In [ ]:
import pandas as pd
from IPython.display import display

TEST_FILES = sorted((DATA_ROOT / 'splits').glob('*/test.csv'))
if not TEST_FILES:
    raise FileNotFoundError(f'No test.csv files in {DATA_ROOT / "splits"}. Check PROJ and Drive access.')

summary = []
for path in TEST_FILES:
    player = path.parent.name
    frame = pd.read_csv(path, dtype={'game_id': str})
    summary.append({
        'player': player,
        'test_games': frame['game_id'].nunique(),
        'test_positions': len(frame),
        'raw_pgn_available': (DATA_ROOT / 'raw' / f'{player}_rapid.pgn').is_file(),
        'maia2_baseline_available': (MAIA2_DIR / f'{player}.csv').is_file(),
    })
display(pd.DataFrame(summary))

selected = DATA_ROOT / 'splits' / PLAYER
for path in (selected / 'test.csv', DATA_ROOT / 'raw' / f'{PLAYER}_rapid.pgn'):
    if not path.is_file():
        raise FileNotFoundError(path)

## 5. Load Maia-3 and confirm it predicts

The first load downloads the pretrained checkpoint into Colab's cache. This cell asks the UCI engine for its top three moves from the starting position at Elo 1500.

In [ ]:
import chess
import chess.engine

engine_command = [sys.executable, '-m', 'maia3.uci', '--model', 'maia3-5m',
                  '--device', DEVICE, '--use-uci-history', '--temperature', '0']
with chess.engine.SimpleEngine.popen_uci(engine_command, timeout=600) as engine:
    engine.ping()
    infos = engine.analyse(chess.Board(), chess.engine.Limit(depth=1), multipv=3,
                           options={'SelfElo': 1500, 'OppoElo': 1500})
    for info in sorted(infos, key=lambda item: item['multipv']):
        print('Rank', info['multipv'], 'move', info['pv'][0].uci())

## 6. S8 timing check on 1,000 positions

Run this before processing every player. The script reports positions per second and an estimated time for 240,000 positions. Sample results are stored separately under `baselines_maia3/pgn/sample_1000/`.

In [ ]:
run_script(S8, '--data-root', DATA_ROOT, '--out-dir', BASELINE_DIR,
           '--maia2-dir', MAIA2_DIR, '--players', PLAYER,
           '--limit', 1000, '--history', 'pgn', '--device', DEVICE)
display(pd.read_csv(BASELINE_DIR / 'pgn' / 'sample_1000' / 'comparison.csv'))

## 7. S8 baseline on every available player

Run this cell when ready for the full evaluation. S8 reconstructs PGN history, handles the mirrored Black positions, and uses P8 for scoring. It prints each player's report and writes prediction CSVs, JSON reports, and `comparison.csv` to Drive.

The summary reports accuracy as fractions from 0 to 1. Missing Maia-2 baseline files leave the comparison pending for those players.

In [ ]:
run_script(S8, '--data-root', DATA_ROOT, '--out-dir', BASELINE_DIR,
           '--maia2-dir', MAIA2_DIR, '--history', 'pgn', '--device', DEVICE)
display(pd.read_csv(BASELINE_DIR / 'pgn' / 'comparison.csv'))

### Optional comparison without move history

Set `RUN_FEN_BASELINE` to `True` to evaluate the same test sets using only the current FEN. Results go into `baselines_maia3/none/`.

In [ ]:
RUN_FEN_BASELINE = False
if RUN_FEN_BASELINE:
    run_script(S8, '--data-root', DATA_ROOT, '--out-dir', BASELINE_DIR,
               '--maia2-dir', MAIA2_DIR, '--history', 'none', '--device', DEVICE)
    display(pd.read_csv(BASELINE_DIR / 'none' / 'comparison.csv'))

## 8. S9 data, loss, and overfit checks

Select `PLAYER` and `TRAIN_FILE` in section 3. These checks use up to 64 positions from that training file. Run the three cells separately and verify each passes before fine-tuning. The overfit check can take time.

S9's experiment defaults enable all parameters and use policy cross-entropy plus 0.1 times outcome cross-entropy. The clock head has no supervised target in these CSVs.

In [ ]:
selected = DATA_ROOT / 'splits' / PLAYER
for path in (selected / TRAIN_FILE, selected / 'val.csv'):
    if not path.is_file():
        raise FileNotFoundError(path)

S9_ARGUMENTS = ['--player', PLAYER, '--data-root', DATA_ROOT,
                '--train-file', TRAIN_FILE, '--out-dir', FINETUNE_DIR,
                '--device', DEVICE, '--batch-size', BATCH_SIZE]
run_script(S9, '--test', 'data', *S9_ARGUMENTS)

In [ ]:
run_script(S9, '--test', 'loss', *S9_ARGUMENTS)

In [ ]:
run_script(S9, '--test', 'overfit', *S9_ARGUMENTS)

## 9. S9 fine-tune and compare before/after

S9 checks that train, validation, and test games are disjoint. It measures the pretrained model on test, fine-tunes on train, selects the best epoch using validation top-1, then scores the selected model on the same test set.

Each improved checkpoint is saved to Drive as `best.pt`. Run outputs also include `before.csv`, `after.csv`, and `report.json`. Restarting this cell starts a new training run and overwrites the outputs for this player/training-file combination.

In [ ]:
run_script(S9, '--test', 'run', *S9_ARGUMENTS,
           '--epochs', EPOCHS, '--lr', LEARNING_RATE)

In [ ]:
import json

RESULT_DIR = FINETUNE_DIR / PLAYER / Path(TRAIN_FILE).stem
report = json.loads((RESULT_DIR / 'report.json').read_text())
comparison = pd.DataFrame([
    {'model': label, 'top1': report[key]['top1']['acc'],
     'top3': report[key]['top3']['acc'],
     'test_games': report[key]['n_games'],
     'test_positions': report[key]['n_positions']}
    for label, key in [('Pretrained Maia-3', 'before'), ('Fine-tuned Maia-3', 'after')]
])
display(comparison)
display(pd.DataFrame(report['history']))
print('Checkpoint:', RESULT_DIR / 'best.pt')

## 10. Record the environment

Run this after any experiment used in the research log. Record the project commit, Maia-3 installation details, Python/PyTorch versions, settings, and GPU information with the results.

In [ ]:
with (RUNS_DIR / 'requirements.txt').open('w') as stream:
    subprocess.run([sys.executable, '-m', 'pip', 'freeze'], stdout=stream, check=True)
commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=REPO, text=True).strip()
package = importlib.metadata.distribution('maia3')
settings = {
    'project_commit': commit, 'repo_ref': REPO_REF, 'python': sys.version,
    'torch': torch.__version__, 'maia3': package.version,
    'maia3_source': package.read_text('direct_url.json'),
    'device': DEVICE, 'project_drive_path': str(PROJ),
    'player': PLAYER, 'train_file': TRAIN_FILE,
    'epochs': EPOCHS, 'batch_size': BATCH_SIZE, 'learning_rate': LEARNING_RATE,
}
(RUNS_DIR / 'environment.json').write_text(json.dumps(settings, indent=2))
if shutil.which('nvidia-smi'):
    with (RUNS_DIR / 'gpu.txt').open('w') as stream:
        subprocess.run(['nvidia-smi'], stdout=stream, check=True)
print('Environment saved to', RUNS_DIR)